In [2]:
import subprocess
from pathlib import Path
import json
import pandas as pd
import os

In [3]:
def xml2csv_multi_call(run_dir: str | Path) -> None:
    xml2csv = Path(os.environ["SUMO_HOME"]) / "tools" / "xml" / "xml2csv.py"

    for xml_file in Path(run_dir).rglob("*.xml"):
        if not xml_file.name.endswith("ess.xml"):
            continue
        subprocess.run(
            ["python", str(xml2csv), str(xml_file)],
            check=True,
        )
        print(f"Converted {xml_file}.")


xml2csv_multi_call(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\increased_run_2026-09-13-17-37-20")
xml2csv_multi_call(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\reduced_run_2026-09-13-16-45-18")
xml2csv_multi_call(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\summer_run_2026-09-13-15-51-44")
xml2csv_multi_call(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07")

Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\increased_run_2026-09-13-17-37-20\65_directory\65_multirun_ess.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\increased_run_2026-09-13-17-37-20\67_directory\67_multirun_ess.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\increased_run_2026-09-13-17-37-20\68_directory\68_multirun_ess.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\increased_run_2026-09-13-17-37-20\70_directory\70_multirun_ess.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\increased_run_2026-09-13-17-37-20\75_directory\75_multirun_ess.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\reduced_run_2026-09-13-16-45-18\65_directory\65_multirun_ess.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo

In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def xml2csv_multi_call(*run_dirs: str | Path, max_workers: int | None = None) -> None:
    xml2csv = Path(os.environ["SUMO_HOME"]) / "tools" / "xml" / "xml2csv.py"

    xml_files = [
        xml_file
        for run_dir in run_dirs
        for xml_file in Path(run_dir).rglob("*.xml")
        if not xml_file.name.endswith("battery.xml")
    ]

    def convert(xml_file: Path) -> Path:
        subprocess.run(["python", str(xml2csv), str(xml_file)], check=True)
        return xml_file

    with ThreadPoolExecutor(max_workers=max_workers or os.cpu_count()) as pool:
        futures = {pool.submit(convert, f): f for f in xml_files}
        for future in as_completed(futures):
            print(f"Converted {futures[future]}.")


xml2csv_multi_call(
    r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output",
    max_workers=10,
)

Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\65_directory\65_multirun_stats.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\65_directory\65_multirun_tripinfo.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\67_directory\67_multirun_chargingstations.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\65_directory\65_multirun_chargingstations.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\67_directory\67_multirun_stats.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\65_directory\65_multirun_ess.xml.
Converted C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best

## Not Working

In [ ]:

from datetime import date
from pathlib import Path

from analysis.output_files import SeedOutputFiles

from energy_storage_system.charging_station import ChargingStation
from energy_storage_system.energy_storage_system import EnergyStorageSystem

PROJECT_ROOT = Path(__file__).resolve().parent
SCENARIO_ROOT = PROJECT_ROOT.parent

SUMO_DIR = SCENARIO_ROOT / "sumo"
SUMO_OUTPUT_DIR = SUMO_DIR / "output"
EBUS_DIR = SCENARIO_ROOT / "eBuS"
FILES_DIR = EBUS_DIR / "files"
PV_DATA_DIR = EBUS_DIR / "pv_estimation/data"

@staticmethod
def run_energy_storage_system( run_dir: Path, start_date: date):
    """
    Build the energy storage system profile from a seed run's SUMO
    chargingstations output (a "<seed>_directory" folder, see
    tools.order_output.order_output) and write the result back as XML.
    Uses the PV data fetched for start_date by run_pvgis_api_call.
    """
    files = SeedOutputFiles(run_dir)
    chargingstations_file = files.get_file("chargingstations")
    output_file = files.output_dir / chargingstations_file.name.replace(
        "_chargingstations.xml", "_ess.xml"
    )
    pv_csv_path = PV_DATA_DIR / f"{start_date}_solar_power_v6_scaled.csv"

    EnergyStorageSystem(
        charging_stations=ChargingStation.from_xml(chargingstations_file),
        ess_factor=4.0,  # each station's ESS = ess_factor * that station's Peak Power (kWh)
        pv_csv_path=pv_csv_path,
        output_path=output_file,
        start_soc=0.2,  # fraction (0.0-1.0) of each station's own ESS capacity
        pv_factor=1.0,  # scales the PV power generated by each station
        grid_charge_max_soc=0.2,  # always draw from grid below this fraction of SoC
        grid_charge_power=50.0,  # power (kW) drawn from grid while below grid_charge_max_soc
        efficiency=0.95,  # battery round-trip efficiency (0.0-1.0)
    ).main()
    print(f"ESS output written to {output_file}")

if __name__ == "__main__":
    run_energy_storage_system(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\storage_run_2026-09-03-14-05-04\67_directory", "2024-08-22")
    run_energy_storage_system(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\storage_run_2026-09-03-14-05-04\68_directory", "2024-08-22")
    run_energy_storage_system(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\storage_run_2026-09-03-14-05-04\70_directory", "2024-08-22")
    run_energy_storage_system(r"G:\Dokumente\Studium\FU Berlin\BeSTeBuS\best-ebus\scenario\sumo\output\storage_run_2026-09-03-14-05-04\75_directory", "2024-08-22")

### Manual ESS caller

In [ ]:

from datetime import date
from pathlib import Path

from analysis.output_files import SeedOutputFiles

from energy_storage_system.charging_station import ChargingStation
from energy_storage_system.energy_storage_system import EnergyStorageSystem

PROJECT_ROOT = Path(__file__).resolve().parent
SCENARIO_ROOT = PROJECT_ROOT.parent

SUMO_DIR = SCENARIO_ROOT / "sumo"
SUMO_OUTPUT_DIR = SUMO_DIR / "output"
EBUS_DIR = SCENARIO_ROOT / "eBuS"
FILES_DIR = EBUS_DIR / "files"
PV_DATA_DIR = EBUS_DIR / "pv_estimation/data"

@staticmethod
def run_energy_storage_system( run_dir: Path, start_date: date):
    """
    Build the energy storage system profile from a seed run's SUMO
    chargingstations output (a "<seed>_directory" folder, see
    tools.order_output.order_output) and write the result back as XML.
    Uses the PV data fetched for start_date by run_pvgis_api_call.
    """
    files = SeedOutputFiles(run_dir)
    chargingstations_file = files.get_file("chargingstations")
    output_file = files.output_dir / chargingstations_file.name.replace(
        "_chargingstations.xml", "_ess.xml"
    )
    pv_csv_path = PV_DATA_DIR / f"{start_date}_solar_power_v6_scaled.csv"

    EnergyStorageSystem(
        charging_stations=ChargingStation.from_xml(chargingstations_file),
        ess_factor=2.0,  # each station's ESS = ess_factor * that station's Peak Power (kWh)
        pv_csv_path=pv_csv_path,
        output_path=output_file,
        start_soc=0.2,  # fraction (0.0-1.0) of each station's own ESS capacity
        pv_factor=1.0,  # scales the PV power generated by each station
        grid_charge_max_soc=0.2,  # always draw from grid below this fraction of SoC
        grid_charge_power=50.0,  # power (kW) drawn from grid while below grid_charge_max_soc
        efficiency=0.95,  # battery round-trip efficiency (0.0-1.0)
    ).main()
    print(f"ESS output written to {output_file}")


if __name__ == "__main__":
    run_energy_storage_system(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\65_directory", "2024-08-22")
    run_energy_storage_system(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\67_directory", "2024-08-22")
    run_energy_storage_system(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\68_directory", "2024-08-22")
    run_energy_storage_system(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\70_directory", "2024-08-22")
    run_energy_storage_system(r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\winter_run_2026-09-13-14-57-07\75_directory", "2024-08-22")


### Trip Lengths

In [9]:
# AI generated on 2026-09-16
import pandas as pd
from lxml import etree

xml_path = r"..\files\postprocessing_input\e_preprocessed_routes.rou.xml"

tree = etree.parse(xml_path)
root = tree.getroot()

routes_dict = {}

for route in root.findall("route"):
    route_id = route.get("id")
    stops = route.findall("stop")
    if stops:
        routes_dict[route_id] = {
            "DURATION": stops[-1].get("until")
        }
        

df = pd.DataFrame.from_dict(routes_dict, orient="index")

df.index.name = "ORIGINAL_TRIP_ID"

print(df)
df.to_csv("../files/preprocessing_input/line_timings.csv")

                 DURATION
ORIGINAL_TRIP_ID         
101_9633           3860.0
101_9646           3980.0
109_9722           1820.0
109_9724           2000.0
110_9735           1970.0
...                   ...
X33_12475          2780.0
X83_12561          2900.0
X83_12570          3140.0
M43_12763          3200.0
M43_12771          3200.0

[76 rows x 1 columns]
